In [1]:
with open(f"data/dialogues_train.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("__eou__ ", "")

print("length of dataset in chars:", len(text))

length of dataset in chars: 5408003


In [2]:
print(text[:1000])

Say , Jim , how about going for a few beers after dinner ? You know that is tempting but is really not good for our fitness . What do you mean ? It will help us to relax . Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? I guess you are right.But what shall we do ? I don't feel like sitting at home . I suggest a walk over to the gym where we can play singsong and meet some of our friends . That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too . Good.Let ' s go now . All right . __eou__
Can you do push-ups ? Of course I can . It's a piece of cake ! Believe it or not , I can do 30 push-ups a minute . Really ? I think that's impossible ! You mean 30 push-ups ? Yeah ! It's easy . If you do exercise everyday , you can make it , too . __eou__
Can you study wit

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [10]:
n_embd = 32

In [4]:
s_i = { ch : i for i, ch in enumerate(chars) }
i_s = { i : ch for i, ch in enumerate(chars) }

encode = lambda s: [s_i[c] for c in s]
decode = lambda l: "".join([i_s[i] for i in l])

print(encode("hello world"))

[67, 64, 71, 71, 74, 1, 82, 74, 77, 71, 63]


In [5]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([5408003]) torch.int64
tensor([50, 60, 84,  1, 13,  1, 41, 68, 72,  1, 13,  1, 67, 74, 82,  1, 60, 61,
        74, 80, 79,  1, 66, 74, 68, 73, 66,  1, 65, 74, 77,  1, 60,  1, 65, 64,
        82,  1, 61, 64, 64, 77, 78,  1, 60, 65, 79, 64, 77,  1, 63, 68, 73, 73,
        64, 77,  1, 30,  1, 56, 74, 80,  1, 70, 73, 74, 82,  1, 79, 67, 60, 79,
         1, 68, 78,  1, 79, 64, 72, 75, 79, 68, 73, 66,  1, 61, 80, 79,  1, 68,
        78,  1, 77, 64, 60, 71, 71, 84,  1, 73, 74, 79,  1, 66, 74, 74, 63,  1,
        65, 74, 77,  1, 74, 80, 77,  1, 65, 68, 79, 73, 64, 78, 78,  1, 15,  1,
        54, 67, 60, 79,  1, 63, 74,  1, 84, 74, 80,  1, 72, 64, 60, 73,  1, 30,
         1, 40, 79,  1, 82, 68, 71, 71,  1, 67, 64, 71, 75,  1, 80, 78,  1, 79,
        74,  1, 77, 64, 71, 60, 83,  1, 15,  1, 35, 74,  1, 84, 74, 80,  1, 77,
        64, 60, 71, 71, 84,  1, 79, 67, 68, 73, 70,  1, 78, 74,  1, 30,  1, 40,
         1, 63, 74, 73,  8, 79,  1, 15,  1, 40, 79,  1, 82, 68, 71, 71,  1, 69,
      

In [6]:
print(decode(data[10000:10130].tolist()))

? your performance was excellent . Thank you for your kindly words . __eou__
Excuse me . Could you please take a picture of us wit


In [7]:
n = int(0.8*len(data))

dtr = data[:n]
dval = data[n:]

In [8]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8 # max context length

def get_batch(split):
    data = dtr if split == "train" else dval
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    return x, y

In [ ]:
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1337)

class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.position_embd = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.token_embd(idx) # (B, T, C)
        pos_emb = self.position_embd(torch.arange(T)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            
            targets = targets.view(-1)
            
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # (B, C)
            
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

In [12]:
net = Bigram()

Xb, yb = get_batch("train")

out, loss = net(Xb, yb)

In [16]:
print(decode(net.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


I'tinyellotharondelink a sid wis eeanfiet ave ong Yong a ur _
I ht tearn I asso plllere ikangiencat 


In [14]:
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3)

In [ ]:
eval_iters = 200

@torch.no_grad()
def estimate_loss():
    out = {}
    net.eval()
    
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        
        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = net(X, y)
            losses[k] = loss.item()
            
        out[split] = losses.mean()
    
    net.train()
    return out

In [15]:
batch_size = 32
max_iter = 10000

for step in range(max_iter):
    Xb, yb = get_batch("train")
    
    logits, loss = net(Xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    
    loss.backward()
    optimizer.step()
    
    if step % 500 == 0:
        print(f'{step} / {max_iter}: {loss.item()}')
        
print("Final loss:", loss)

0 / 10000: 4.813259124755859
500 / 10000: 2.6368093490600586
1000 / 10000: 2.4828412532806396
1500 / 10000: 2.478458881378174
2000 / 10000: 2.2799761295318604
2500 / 10000: 2.2996878623962402
3000 / 10000: 2.367955446243286
3500 / 10000: 2.4136528968811035
4000 / 10000: 2.3324031829833984
4500 / 10000: 2.284285306930542
5000 / 10000: 2.2446436882019043
5500 / 10000: 2.340369462966919
6000 / 10000: 2.484506130218506
6500 / 10000: 2.329451560974121
7000 / 10000: 2.307426929473877
7500 / 10000: 2.385939121246338


KeyboardInterrupt: 

In [14]:

print(decode(net.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


Hatiththy . melyoure tte ’ bDou ! . It ? cale to h fof Bulandok o we ghouatis kn Che Okis V8 an . No


In [84]:
print(out.shape)

torch.Size([4, 8, 100])
